MOVIE RECOMMENDATION SYSTEM

In [ ]:
#Import Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Flatten, Concatenate, Dense
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

In [ ]:
# Load MovieLens dataset 
df = pd.read_csv("ratings.csv")  

In [ ]:
# Convert userId and movieId to index values
user_ids = df['userId'].unique().tolist()
movie_ids = df['movieId'].unique().tolist()

In [ ]:
user_to_index = {x: i for i, x in enumerate(user_ids)}
movie_to_index = {x: i for i, x in enumerate(movie_ids)}

In [ ]:
df['user'] = df['userId'].map(user_to_index)
df['movie'] = df['movieId'].map(movie_to_index)

In [ ]:
# Train-test split
train, test = train_test_split(df[['user', 'movie', 'rating']], test_size=0.2, random_state=42)

In [ ]:
n_users = len(user_ids)
n_movies = len(movie_ids)

In [ ]:
train.shape, test.shape

In [ ]:
# Deep Learning Recommendation Model

In [ ]:
# Input
user_input = Input(shape=(1,), name='user_input')
movie_input = Input(shape=(1,), name='movie_input')

In [ ]:
# Embedding
user_embedding = Embedding(input_dim=n_users, output_dim=50, name='user_embedding')(user_input)
movie_embedding = Embedding(input_dim=n_movies, output_dim=50, name='movie_embedding')(movie_input)

In [ ]:
# Flatten
user_vec = Flatten()(user_embedding)
movie_vec = Flatten()(movie_embedding)

In [ ]:
# Concatenate user and movie embedding
concat = Concatenate()([user_vec, movie_vec])

In [ ]:
# Dense layer
dense = Dense(128, activation='relu')(concat)
dense = Dense(64, activation='relu')(dense)
output = Dense(1)(dense)

In [ ]:
# Model
model = Model(inputs=[user_input, movie_input], outputs=output)
model.compile(optimizer=Adam(0.001), loss='mse')

In [ ]:
model.summary()

In [ ]:
# Model Training

In [ ]:
# Preparing training data
train_user = train['user'].values
train_movie = train['movie'].values
train_rating = train['rating'].values

In [ ]:
test_user = test['user'].values
test_movie = test['movie'].values
test_rating = test['rating'].values

In [ ]:
# Train
model.fit([train_user, train_movie], train_rating,
          validation_data=([test_user, test_movie], test_rating),
          epochs=5, batch_size=64)

In [ ]:
# Predict Rating for a User-Movie Pair

In [ ]:
# Predict rating
user_idx = user_to_index[1]
movie_idx = movie_to_index[1029]

In [ ]:
predicted = model.predict([[user_idx], [movie_idx]])
print("Predicted rating:", predicted[0][0])

In [ ]:
# Recommend Top-N(N is number input) Movies for a User

In [ ]:
def recommend_movies(user_id, top_n=5):
    user_idx = user_to_index[user_id]
    all_movie_indices = np.array(list(movie_to_index.values()))
    
    user_array = np.array([user_idx] * len(all_movie_indices))
    preds = model.predict([user_array, all_movie_indices])
    
    top_indices = preds.reshape(-1).argsort()[-top_n:][::-1]
    recommended_movie_ids = [movie_ids[i] for i in top_indices]
    
    return recommended_movie_ids

In [ ]:
recommend_movies(1)